In [0]:
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession

def extrair_e_carregar_cambio_usd():
    """
    EXTRACT & LOAD (ELT): Busca o histórico completo do dólar comercial (USD/BRL)
    na API oficial do Banco Central do Brasil (BCB) e salva bruto na Bronze.
    """
    # 1. Configuração de datas dinâmicas para cobrir o histórico completo
    data_inicial = "01-01-2011"
    data_final = datetime.today().strftime("%m-%d-%Y") # Formato MM-DD-AAAA exigido pela API do BCB
    
    # Ajustado para usar o endpoint oficial OData do BCB
    url = (
        "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
        f"@dataInicial='{data_inicial}'&@dataFinalCotacao='{data_final}'"
        "&$top=10000&$format=json"
    )
    
    # 💡 CORREÇÃO DO ERRO 403: Simulando uma requisição vinda de um navegador real
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
        "Connection": "keep-alive"
    }
    
    print(f"🔄 [Extract] Buscando cotações de USD/BRL de {data_inicial} até {data_final}...")
    try:
        # Enviamos os headers criados junto com a requisição GET
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        dados_brutos = response.json()
    except Exception as e:
        raise RuntimeError(f"❌ Falha ao consumir a API do Banco Central: {e}")
        
    # O BCB retorna os registros dentro da chave 'value'
    lista_cotacoes = dados_brutos.get("value", [])
    
    if not lista_cotacoes:
        print("⚠️ Nenhum dado retornado pela API do BCB.")
        return
        
    # 2. Cria o DataFrame do Pandas temporário
    df_pandas = pd.DataFrame(lista_cotacoes)
    
    # Tratamento simples de data apenas para compatibilidade de fuso horário no Spark
    df_pandas["dataHoraCotacao"] = pd.to_datetime(df_pandas["dataHoraCotacao"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    
    # 3. Converte para Spark DataFrame
    df_spark = spark.createDataFrame(df_pandas)
    
    # 4. Garante que o schema bronze exista
    print("📁 Garantindo que o schema 'bronze' exista...")
    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
    
    # 5. [Load] Grava o dado bruto na tabela Bronze como Delta Lake
    print("💾 [Load] Salvando dados brutos na tabela bronze.cambio_usd_brl_raw...")
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze.cambio_usd_brl_raw")
        
    print("✅ Ingestão Bronze de USD concluída com sucesso!")

# Executa o pipeline
extrair_e_carregar_cambio_usd()

In [0]:
%sql
SELECT * FROM bronze.cambio_usd_brl_raw ORDER BY dataHoraCotacao ASC LIMIT 5;